# Deterministic oscillatory game with the original map-DTB code

This standalone implementation of the oscillatory game uses the original lower-level files `DTB_rep/dtb.py` and `DTB_rep/run_game_dtb.py`; it does **not** import the newer `game_dtb` package.

The neural network represents a pushforward map $T_\theta:z\mapsto x$. DTB advances that map in its parameter-tangent space, while RK4 directly solves the deterministic particle ODE and provides an independent reference. In Colab, select **Runtime > Change runtime type > T4 GPU**, then run the cells from top to bottom.

In [1]:
# Clone the branch containing this notebook. If the runtime already has the
# repository, pull the latest committed version instead of cloning again.
import os, pathlib, subprocess, sys

REPO_DIR = pathlib.Path("/content/dtb-colab-experiments")
BRANCH = "codex/game-dynamics-dtb"
if REPO_DIR.exists():
    subprocess.run(["git", "-C", str(REPO_DIR), "pull", "--ff-only"], check=True)
else:
    subprocess.run([
        "git", "clone", "-q", "--depth", "1", "--branch", BRANCH,
        "https://github.com/sun-mengwei/dtb-colab-experiments.git", str(REPO_DIR),
    ], check=True)

# dtb.py and run_game_dtb.py live in DTB_rep.
os.chdir(REPO_DIR / "DTB_rep")
sys.path.insert(0, str(REPO_DIR / "DTB_rep"))
print("Using:", pathlib.Path.cwd())

Using: /content/dtb-colab-experiments/DTB_rep


## Deterministic game

For $x=(x_1,x_2)$, define

$$\Phi(x)=-\frac{\lambda}{2}(x_1^2+x_2^2)-\frac{\gamma}{2}(x_1-x_2)^2+\frac{\varepsilon}{\omega}[\cos(\omega x_1)+\cos(\omega x_2)].$$

The deterministic game dynamics are $\dot x=b(x)=\nabla\Phi(x)$, where

$$b(x)=\begin{bmatrix}-\lambda x_1-\gamma(x_1-x_2)-\varepsilon\sin(\omega x_1)\\-\lambda x_2-\gamma(x_2-x_1)-\varepsilon\sin(\omega x_2)\end{bmatrix}.$$

There is no diffusion or density score in this notebook, so the DTB step needs parameter derivatives but no spatial-density derivatives.

## Analytical equilibria

Let $m=(x_1+x_2)/2$ and $d=(x_1-x_2)/2$. The exact equilibrium conditions are

$$\lambda m+\varepsilon\sin(\omega m)\cos(\omega d)=0,$$

$$(\lambda+2\gamma)d+\varepsilon\cos(\omega m)\sin(\omega d)=0.$$

The origin is always an equilibrium. Its Jacobian eigenvalues are $-\lambda-\varepsilon\omega$ and $-\lambda-2\gamma-\varepsilon\omega$. Both are negative for the parameters below, so the origin is locally asymptotically stable. Other stable roots create the additional attraction basins visible in the snapshots.

In [2]:
import math, time
from pprint import pprint

import matplotlib.pyplot as plt
import numpy as np
import torch

# Some Colab PyTorch builds fail to lazily expose this internal submodule
# when torch.func.jacrev is first used. This guarded import is compatibility
# plumbing only; it is not an additional step in the DTB method.
try:
    import torch._dynamo.compiled_autograd
except (AttributeError, ImportError):
    pass

from dtb import device, flat_params, jform_solve
from run_game_dtb import (
    ResidualMLPMap, CurrentGameMap, game_dtb_basis_matrices, fit_map_to_target,
)

# One parameter pair. Change these values only after validating this run.
LAMBDA = 0.5
GAMMA = 0.2
EPSILON = 0.5
OMEGA = 4.0 * math.pi

PARTICLES = 500
STEPS = 1000
STEP_SIZE = 1.0 / STEPS       # final time T = 1
SEED = 2026
MODEL_SEED = 91

WIDTH = 16
DEPTH = 2
BASIS_SIZE = None             # None uses all trainable parameters
JACOBIAN_CHUNK = 250          # memory batching only; it does not change J
SVD_RTOL = 1e-3
SOLVER = "lstsq"

# Hold one tangent basis for 100 physical steps, then compress the
# accumulated map into the MLP. Set this to 0 to disable refitting.
REFIT_INTERVAL = 100
REFIT_OPTIMIZER_STEPS = 200
REFIT_LEARNING_RATE = 1e-3
REFIT_SAMPLES = 1024
REFIT_BATCH_SIZE = 256

DTYPE = torch.float32
DEVICE = device()
if DEVICE.type == "cuda":
    torch.set_float32_matmul_precision("high")

pprint({
    "lambda": LAMBDA, "gamma": GAMMA, "epsilon": EPSILON,
    "omega": f"{OMEGA / math.pi:g} pi",
    "particles": PARTICLES, "steps": STEPS, "step_size": STEP_SIZE,
    "final_time": STEPS * STEP_SIZE,
    "network": f"ResidualMLPMap(width={WIDTH}, depth={DEPTH})",
    "basis_size": "all parameters" if BASIS_SIZE is None else BASIS_SIZE,
    "refit_interval": REFIT_INTERVAL,
    "planned_refits": STEPS // REFIT_INTERVAL if REFIT_INTERVAL else 0,
    "device": str(DEVICE),
})
if DEVICE.type != "cuda":
    print("GPU not detected. In Colab choose Runtime > Change runtime type > T4 GPU.")

{'basis_size': 'all parameters',
 'device': 'cpu',
 'epsilon': 0.5,
 'final_time': 1.0,
 'gamma': 0.2,
 'lambda': 0.5,
 'network': 'ResidualMLPMap(width=16, depth=2)',
 'omega': '4 pi',
 'particles': 500,
 'planned_refits': 10,
 'refit_interval': 100,
 'step_size': 0.001,
 'steps': 1000}
GPU not detected. In Colab choose Runtime > Change runtime type > T4 GPU.


## Experiment-specific functions

RK4 and DTB start from the same fixed labels $z_i\sim U([-1,1]^2)$. RK4 directly advances the particles. DTB freezes $\theta$, a parameter subset, and its tangent matrix for each block, then accumulates coefficients in that fixed tangent space.

In [3]:
def potential(x):
    x1, x2 = x.unbind(dim=-1)
    return (
        -0.5 * LAMBDA * (x1.square() + x2.square())
        -0.5 * GAMMA * (x1 - x2).square()
        + (EPSILON / OMEGA) * (torch.cos(OMEGA * x1) + torch.cos(OMEGA * x2))
    )


def velocity(x):
    # Deterministic pseudo-gradient b(x) = grad Phi(x).
    x1, x2 = x.unbind(dim=-1)
    v1 = -LAMBDA * x1 - GAMMA * (x1 - x2) - EPSILON * torch.sin(OMEGA * x1)
    v2 = -LAMBDA * x2 - GAMMA * (x2 - x1) - EPSILON * torch.sin(OMEGA * x2)
    return torch.stack((v1, v2), dim=-1)


def make_initial_particles():
    # Draw on CPU for reproducibility, then move the fixed labels to the GPU.
    generator = torch.Generator().manual_seed(SEED)
    labels = 2.0 * torch.rand(PARTICLES, 2, generator=generator, dtype=DTYPE) - 1.0
    return labels.to(DEVICE)


def solve_with_rk4(initial_particles):
    # RK4 is reference-only. It never enters the DTB parameter update.
    x = initial_particles.detach().clone()
    history = [x.cpu().numpy().copy()]
    for _ in range(STEPS):
        k1 = velocity(x)
        k2 = velocity(x + 0.5 * STEP_SIZE * k1)
        k3 = velocity(x + 0.5 * STEP_SIZE * k2)
        k4 = velocity(x + STEP_SIZE * k3)
        x = x + (STEP_SIZE / 6.0) * (k1 + 2.0 * k2 + 2.0 * k3 + k4)
        history.append(x.cpu().numpy().copy())
    return np.stack(history)


def solve_with_map_dtb(labels):
    torch.manual_seed(MODEL_SEED)
    model = ResidualMLPMap(
        dim=2, width=WIDTH, depth=DEPTH, activation="tanh", dtype=DTYPE
    ).to(DEVICE)

    # flat_params provides the vector theta and the metadata needed by the
    # functional parameter-Jacobian calculation.
    theta_block, structure, _ = flat_params(model)
    parameter_count = theta_block.numel()
    basis_count = parameter_count if BASIS_SIZE is None else min(BASIS_SIZE, parameter_count)
    generator_device = DEVICE.type if DEVICE.type == "cuda" else "cpu"
    basis_generator = torch.Generator(device=generator_device).manual_seed(SEED + 20_000)

    def choose_basis():
        # The full basis is recommended for the first validation because the
        # residual map has an exactly-zero output layer at initialization.
        if basis_count == parameter_count:
            return torch.arange(parameter_count, device=DEVICE)
        return torch.randperm(
            parameter_count, device=DEVICE, generator=basis_generator
        )[:basis_count].sort().values

    def start_block():
        # theta and the selected tangent basis remain frozen within a block.
        theta, current_structure, _ = flat_params(model)
        if current_structure != structure:
            raise RuntimeError("The model parameter structure changed.")
        selected = choose_basis()
        y_base, tangent, tangent_flat = game_dtb_basis_matrices(
            theta, selected, labels, model, structure, chunk=JACOBIAN_CHUNK
        )
        # Nothing in the time loop differentiates through this basis, so
        # detaching it prevents an unnecessary autograd graph from growing.
        return (
            theta, selected, y_base.detach(), tangent.detach(),
            tangent_flat.detach(),
            torch.zeros(selected.numel(), device=DEVICE, dtype=DTYPE),
        )

    theta_block, selected, y_base, tangent, tangent_flat, accumulated_alpha = start_block()
    steps_in_block = 0
    residuals, refit_steps, refit_rmse = [], [], []
    history = [labels.cpu().numpy().copy()]
    milestones = {math.ceil(STEPS * fraction / 10) for fraction in range(1, 11)}
    start_time = time.perf_counter()

    print(f"trainable parameters={parameter_count}, tangent basis={basis_count}")
    for step in range(STEPS):
        # Current accumulated map: T_k(z) = T_block(z) + h J(z) s_k.
        x = y_base + STEP_SIZE * torch.einsum(
            "ndm,m->nd", tangent, accumulated_alpha
        )
        target_flat = velocity(x).detach().reshape(-1)

        # Core DTB solve: alpha_k = argmin ||J alpha - b(T_k(z))||.
        alpha = jform_solve(
            tangent_flat, target_flat, rtol=SVD_RTOL, method=SOLVER
        ).detach()
        projected_flat = tangent_flat @ alpha
        residual = torch.linalg.norm(projected_flat - target_flat) / (
            torch.linalg.norm(target_flat) + 1e-30
        )
        residuals.append(float(residual))

        # Holding J fixed makes s_{k+1} = s_k + alpha_k the whole DTB update.
        accumulated_alpha = accumulated_alpha + alpha
        steps_in_block += 1
        x_next = y_base + STEP_SIZE * torch.einsum(
            "ndm,m->nd", tangent, accumulated_alpha
        )
        history.append(x_next.cpu().numpy().copy())

        if REFIT_INTERVAL and steps_in_block == REFIT_INTERVAL:
            # CurrentGameMap packages the accumulated tangent state as a
            # callable target. fit_map_to_target compresses it into all MLP
            # parameters; it does not fit the network directly to b(x).
            target_map = CurrentGameMap(
                theta_block, selected, accumulated_alpha, STEP_SIZE,
                model, structure, chunk=JACOBIAN_CHUNK,
            )
            rmse = fit_map_to_target(
                model, target_map, dim=2, low=-1.0, high=1.0,
                n_samples=REFIT_SAMPLES, steps=REFIT_OPTIMIZER_STEPS,
                lr=REFIT_LEARNING_RATE, batch_size=REFIT_BATCH_SIZE,
            )
            refit_steps.append(step + 1)
            refit_rmse.append(rmse)
            # The refitted network becomes the actual map for the next block.
            # Replace the boundary snapshot so history follows that same map.
            history[-1] = model(labels).detach().cpu().numpy().copy()
            steps_in_block = 0
            if step + 1 < STEPS:
                (theta_block, selected, y_base, tangent, tangent_flat,
                 accumulated_alpha) = start_block()

        completed = step + 1
        if completed in milestones:
            if DEVICE.type == "cuda":
                torch.cuda.synchronize()
            elapsed = time.perf_counter() - start_time
            eta = elapsed * (STEPS - completed) / completed
            last_refit = f"{refit_rmse[-1]:.2e}" if refit_rmse else "n/a"
            print(
                f"DTB {100 * completed / STEPS:5.1f}% | step {completed}/{STEPS} | "
                f"t={completed * STEP_SIZE:.3f} | residual={float(residual):.3e} | "
                f"refits={len(refit_steps)} | last refit RMSE={last_refit} | "
                f"elapsed={elapsed / 60:.1f} min | ETA={eta / 60:.1f} min"
            )

    return np.stack(history), np.asarray(residuals), refit_steps, refit_rmse


def plot_particle_snapshots(history, title):
    indices = np.linspace(0, STEPS, 5, dtype=int)
    fig, axes = plt.subplots(1, 5, figsize=(15, 3), sharex=True, sharey=True)
    for axis, index in zip(axes, indices):
        points = history[index]
        axis.scatter(points[:, 0], points[:, 1], s=8, alpha=0.6)
        axis.set_title(f"t = {index * STEP_SIZE:.2f}")
        axis.set_xlim(-1.1, 1.1)
        axis.set_ylim(-1.1, 1.1)
        axis.set_aspect("equal")
        axis.grid(alpha=0.2)
        axis.set_xlabel(r"$x_1$")
    axes[0].set_ylabel(r"$x_2$")
    fig.suptitle(title)
    plt.tight_layout()
    plt.show()

## Two small checks

The first check verifies the experiment-specific derivative $b=\nabla\Phi$. The second verifies that `ResidualMLPMap` begins as the identity, so the neural map initially represents the requested uniform distribution.

In [4]:
probe = torch.tensor(
    [[0.17, -0.31], [-0.62, 0.44]], device=DEVICE, dtype=DTYPE, requires_grad=True
)
autograd_velocity = torch.autograd.grad(potential(probe).sum(), probe)[0]
derivative_error = (autograd_velocity - velocity(probe)).abs().max().item()

torch.manual_seed(MODEL_SEED)
identity_model = ResidualMLPMap(
    dim=2, width=WIDTH, depth=DEPTH, activation="tanh", dtype=DTYPE
).to(DEVICE)
identity_error = (identity_model(probe.detach()) - probe.detach()).abs().max().item()

print(f"max |autograd grad(Phi) - velocity| = {derivative_error:.3e}")
print(f"max |initial map(z) - z| = {identity_error:.3e}")
assert derivative_error < 2e-6 and identity_error == 0.0

max |autograd grad(Phi) - velocity| = 1.490e-08
max |initial map(z) - z| = 0.000e+00


## Run RK4 and map-DTB

Both deterministic methods start from exactly the same uniform particles and use the same physical step size. RK4 has much smaller integration error and is reference-only; the DTB update remains explicit Euler in the learned tangent space.

In [ ]:
initial_particles = make_initial_particles()
ode_history = solve_with_rk4(initial_particles)
dtb_history, projection_residual, refit_steps, refit_rmse = solve_with_map_dtb(
    initial_particles
)
times = np.arange(STEPS + 1) * STEP_SIZE
origin_eigenvalues = np.array([
    -LAMBDA - EPSILON * OMEGA,
    -LAMBDA - 2.0 * GAMMA - EPSILON * OMEGA,
])
print(
    f"Origin stable: {bool(origin_eigenvalues.max() < 0)}; "
    f"Jacobian eigenvalues={origin_eigenvalues}; "
    f"refits={len(refit_steps)} at steps {refit_steps}; "
    f"refit RMSE={refit_rmse}."
)

## Projection residual over time

The relative residual is $\lVert J\alpha-b\rVert/\lVert b\rVert$. Values near zero mean the frozen tangent basis represents the deterministic game velocity well; values near one mean it does not.

In [ ]:
plt.figure(figsize=(6, 3.5))
plt.semilogy(times[1:], np.maximum(projection_residual, 1e-12), label="map-DTB residual")
for index, step in enumerate(refit_steps):
    plt.axvline(times[step], color="tab:orange", alpha=0.25,
                label="NN refit" if index == 0 else None)
plt.xlabel("time")
plt.ylabel("relative projection residual")
plt.legend()
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()

## Standard RK4 ODE reference

In [ ]:
plot_particle_snapshots(ode_history, "Standard RK4 particles")

## Deterministic map-DTB approximation

In [ ]:
plot_particle_snapshots(dtb_history, "Deterministic map-DTB particles")